# 개별종목 조합J — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합J 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합J의 피처 값만 지정합니다.
import json

COMBINATION = 'J'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합J 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4348,0.5012,-0.0664,0.3668,0.3695,0.0600,0.3587,0.2665,0.3417
1,2,balanced,980,20150123,20150421,0.3655,0.3978,-0.0323,0.3497,0.3535,0.0324,0.3596,0.3026,0.3371
2,3,balanced,1210,20151228,20160328,0.3580,0.3762,-0.0182,0.3575,0.3588,0.0379,0.3575,0.3819,0.3655
3,4,balanced,1439,20161202,20170228,0.3901,0.4617,-0.0716,0.3458,0.3499,0.0274,0.3564,0.2840,0.3342
4,5,balanced,1669,20171113,20180207,0.3824,0.3901,-0.0077,0.3687,0.3703,0.0579,0.3700,0.3303,0.3591
5,6,balanced,1899,20181024,20190118,0.3948,0.3725,0.0223,0.3946,0.3967,0.0964,0.3989,0.4185,0.4023
6,7,balanced,2129,20190930,20191224,0.4255,0.4781,-0.0526,0.3644,0.3702,0.0676,0.3767,0.2760,0.3441
7,8,balanced,2359,20200902,20201130,0.3707,0.3476,0.0231,0.3700,0.3728,0.0584,0.3748,0.3998,0.3797
8,9,balanced,2589,20210806,20211105,0.3800,0.3916,-0.0117,0.3676,0.3704,0.0597,0.3681,0.3536,0.3667
9,10,balanced,2818,20220714,20221012,0.3623,0.3454,0.0169,0.3610,0.3621,0.0429,0.3633,0.3529,0.3587


,OOS 폴드 평균
accuracy,0.3830
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0138
macro_f1,0.3643
balanced_accuracy,0.3668
mcc,0.0529
pr_auc_macro_ovr,0.3686
down_recall,0.3396
core_harmonic_mean,0.3592


재실행 명령: python scripts/run_stock_model_experiment.py
